# tensor-ml Quickstart Tutorial

Welcome to **tensor-ml** — a backend-agnostic tensor machine-learning library
with first-class NumPy and PyTorch support.

This notebook covers:

1. **Installation** and imports
2. **Tensor operations** — element-wise ops and tensor products
3. **T-LARS** — fit, predict, score, and parameter introspection
4. **Backend switching** — NumPy ↔ PyTorch (when available)

## 1. Installation

```bash
# Recommended (using uv):
uv sync --all-extras

# Or with pip from the repo root:
pip install -e .

# With PyTorch support:
pip install -e ".[torch]"
```

In [1]:
import numpy as np
from tensor_ml import (
    # Enums
    BackendType,
    # Tensor ops
    TensorOps, NumpyOps, TensorOpsFactory,
    # Tensor products
    TensorProducts, NumpyTensorProducts, TensorProductsFactory,
    # Models
    BaseTensorModel, MultilinearModel, TLARS,
)
print("tensor-ml imported successfully ✓")

tensor-ml imported successfully ✓


## 2. Tensor Operations

`TensorOps` provides **element-wise** operations (norm, flatten, sign, …).
`TensorProducts` provides **multi-linear** products (Kronecker, Khatri-Rao, full
multilinear product, …).

Both have factory classes that return the correct backend automatically.

In [2]:
# ── Element-wise ops via the factory ──────────────────────────────
ops = TensorOpsFactory.get(BackendType.NUMPY)

x = np.array([3.0, 4.0])
print("norm:", ops.norm(x))                  # 5.0
print("sign:", ops.sign(np.array([-2, 0, 5])))  # [-1, 0, 1]

# Column-normalise a matrix
D = np.random.randn(5, 3)
D_normed = ops.normalize(D)
col_norms = np.linalg.norm(D_normed, axis=0)
print("Column norms after normalize:", np.round(col_norms, 10))  # [1, 1, 1]

norm: 5.0
sign: [-1  0  1]
Column norms after normalize: [1. 1. 1.]


In [3]:
# ── Tensor products (static facade — auto-detects backend) ──────
A = np.array([[1, 2], [3, 4]])
B = np.array([[0, 5], [6, 7]])

# Kronecker product: A ⊗ B
kron = TensorProducts.kronecker_product([A, B])
print("Kronecker product shape:", kron.shape)
print(kron)

# Khatri-Rao product (column-wise Kronecker)
kr = TensorProducts.khatri_rao_product([A, B])
print("\nKhatri-Rao product shape:", kr.shape)
print(kr)

# Full multilinear product: Y = X ×₁ A ×₂ B
X = np.random.randn(2, 2)
Y = TensorProducts.full_multilinear_product(X, [A, B])
print("\nFull multilinear product shape:", Y.shape)

Kronecker product shape: (4, 4)
[[ 0  0  5 10]
 [ 0  0 15 20]
 [ 6 12  7 14]
 [18 24 21 28]]

Khatri-Rao product shape: (4, 2)
[[ 0 10]
 [ 0 20]
 [ 6 14]
 [18 28]]

Full multilinear product shape: (2, 2)


## 3. T-LARS: Sparse Tensor Recovery

The **Tensor Least Angle Regression (T-LARS)** algorithm recovers a sparse coefficient tensor from a measurement vector and a Kronecker-structured dictionary.

**Workflow:** `fit` → `predict` → `score`

In [4]:
from tensor_ml.tensor_models import TLARS, TLARSConfig

# ── Build a small synthetic problem ─────────────────────────────
# Random per-mode dictionaries
D1 = np.random.randn(8, 16)     # mode-1: 8 signal rows, 16 atoms
D2 = np.random.randn(8, 16)     # mode-2: 8 signal rows, 16 atoms

# Ground-truth sparse coefficient tensor
C_true = np.zeros((16, 16))
rng = np.random.default_rng(42)
for _ in range(5):
    i, j = rng.integers(0, 16, size=2)
    C_true[i, j] = rng.standard_normal()

# Target tensor: Y = D1 @ C_true @ D2.T  (full multilinear product)
Y = D1 @ C_true @ D2.T

print(f"Target tensor shape: {Y.shape}")
print(f"True sparsity:       {np.count_nonzero(C_true)}")

Target tensor shape: (8, 8)
True sparsity:       5


In [5]:
# ── Configure and run T-LARS ─────────────────────────────────────
model = TLARS(
    tolerance=0.01,
    debug_mode=True,          # emit per-iteration DEBUG logs
    l0_mode=True,             # L0 (greedy) selection
)
print("Config:", model.get_params())
print(repr(model))

# Fit — pass per-mode dictionaries and the target tensor
import logging
logging.basicConfig(level=logging.DEBUG, format="%(message)s")

model.fit(factor_matrices=[D1, D2], Y=Y)

# Predict — reconstructs the full tensor
Y_hat = model.predict([D1, D2])

# Score — R² on training data
r2 = model.score([D1, D2], Y)
print(f"\nR² score: {r2:.6f}")
print(f"Iterations: {model.n_iter_}")

TLARS setup: backend=BackendType.NUMPY, tensor_shape=(8, 8), core_tensor_shape=[16, 16], total_columns=256, orthogonal=False, l0_mode=True, mask_type=KP
TLARS normalisation: tensor_norm=14.9809, initial ||r||=1
iter 1: n_active=1, lambda=0.693967
iter 1: adding column 214
iter 2: n_active=2, lambda=0.575724
iter 2: adding column 193
iter 3: n_active=3, lambda=0.391735
iter 3: adding column 40
iter 4: n_active=4, lambda=0.239885
iter 4: adding column 56
iter 5: n_active=5, lambda=0.112775
iter 5: adding column 203
iter 6: n_active=6, lambda=0.101281
iter 6: stopping — tolerance (||r||=1.08966e-14, n_active=6)
TLARS finished: iterations=6, final ||r||=1.08966e-14, n_active=6


Config: {'tolerance': 0.01, 'l0_mode': True, 'mask_type': 'KP', 'debug_mode': True, 'active_coefficients': 1000000, 'iterations': 1000000, 'precision_factor': 5, 'show_progress': False, 'backend': None, 'device': None}
TLARS(tolerance=0.01, l0_mode=True, debug_mode=True)

R² score: 1.000000
Iterations: 6


In [6]:
# ── Inspect the recovered coefficients ───────────────────────────
C_hat = model.coef_tensor_    # sparse coefficient tensor

print(f"Recovered sparsity: {np.count_nonzero(C_hat)}")
print(f"Active atoms:       {len(model.active_columns_)}")
print(f"Final residual:     {model.norm_r_[-1]:.6f}")

Recovered sparsity: 6
Active atoms:       6
Final residual:     0.000000


## 4. Parameter Introspection

T-LARS follows a scikit-learn-like interface. You can inspect and modify parameters at any time.

In [7]:
# get_params returns a dict of current configuration
params = model.get_params()
for k, v in sorted(params.items()):
    print(f"  {k}: {v}")

# set_params lets you change parameters (returns self for chaining)
model.set_params(debug_mode=False, l0_mode=False)
print("\nAfter set_params:")
print(f"  debug_mode = {model.get_params()['debug_mode']}")
print(f"  l0_mode    = {model.get_params()['l0_mode']}")

  active_coefficients: 1000000
  backend: None
  debug_mode: True
  device: None
  iterations: 1000000
  l0_mode: True
  mask_type: KP
  precision_factor: 5
  show_progress: False
  tolerance: 0.01

After set_params:
  debug_mode = False
  l0_mode    = False


## 5. Backend Switching (NumPy ↔ PyTorch)

tensor-ml ships with a NumPy backend by default. If PyTorch is installed, the library auto-detects the array type and dispatches to GPU-accelerated routines.

```python
import torch, numpy as np
from tensor_ml.tensor_ops import TensorProducts

# NumPy input → NumPy backend
A_np = np.eye(3)
TensorProducts.kronecker_product([A_np, A_np])  # uses NumpyTensorProducts

# PyTorch input → PyTorch backend
A_pt = torch.eye(3)
TensorProducts.kronecker_product([A_pt, A_pt])  # uses TorchTensorProducts
```

The same applies to `TensorOps` element-wise operations. No code changes required — just pass the right array type.

## 6. Next Steps

- **Image reconstruction example:** See `docs/examples/tlars_image_reconstruction.ipynb` for a visual demo with DCT dictionaries and per-iteration snapshots.
- **API reference:** See `docs/api_reference.md` for full class and method documentation.
- **User guide:** See `docs/user_guide.md` for in-depth explanations of concepts and architecture.